<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.2_Praktikum_Image_Reading_dan_Manipulation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.2: Praktikum Image Reading dan Manipulation
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Membaca dan memvalidasi berkas citra secara aman dengan penanganan galat `NoneType`.
2. Melakukan pengirisan area minat (*Region of Interest* / ROI) pada citra tandan buah sawit.
3. Mengimplementasikan koreksi rotasi dan translasi geometri afina menggunakan `cv2.warpAffine`.
4. Membandingkan algoritma interpolasi resolusi (`INTER_AREA`, `INTER_LINEAR`, `INTER_NEAREST`) dan mengimplementasikan letterboxing.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Citra Sintetis Meja Sortasi dan Pemotongan ROI
Mensimulasikan tandan buah sawit di meja konveyor dan mengekstrak area minat (ROI) menggunakan slicing NumPy.

In [ ]:
# Membuat citra konveyor abu-abu gelap 600x800
conveyor = np.full((600, 800, 3), 45, dtype=np.uint8)

# Menggambar tandan buah sawit miring 30 derajat
center_palm = (420, 310)
cv2.ellipse(conveyor, center_palm, (160, 95), 30, 0, 360, (25, 115, 230), -1)
cv2.circle(conveyor, (420, 310), 25, (10, 60, 160), -1)

# Simpan citra sintetis
cv2.imwrite('conveyor_sample.jpg', conveyor)

# Pemotongan ROI Tandan Buah (Y: 180..440, X: 220..620)
y1, y2 = 180, 440
x1, x2 = 220, 620
roi_palm = conveyor[y1:y2, x1:x2].copy()

print(f'Dimensi Citra Konveyor Asli : {conveyor.shape}')
print(f'Dimensi ROI Buah Terpotong  : {roi_palm.shape}')


## 2. Koreksi Rotasi Geometri Affine (Pelurusan Posisi Tandan)
Membangun matriks rotasi afina untuk meluruskan orientasi tandan sebesar -30 derajat.

In [ ]:
h_roi, w_roi = roi_palm.shape[:2]
center_roi = (w_roi // 2, h_roi // 2)

# Matriks rotasi 2x3
M_rot = cv2.getRotationMatrix2D(center=center_roi, angle=-30, scale=1.0)
roi_straight = cv2.warpAffine(roi_palm, M_rot, (w_roi, h_roi), borderMode=cv2.BORDER_CONSTANT, borderValue=(45, 45, 45))

print('Matriks Transformasi Afina (2x3):\n', M_rot)


## 3. Penskalaan dengan Letterboxing (Mempertahankan Aspek Rasio)
Mengubah ukuran ROI ke dimensi model AI standar 224x224 dengan padding abu-abu netral.

In [ ]:
def letterbox_resize(img, target_size=(224, 224), pad_color=(45, 45, 45)):
    h, w = img.shape[:2]
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    
    nw, nh = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    
    canvas = np.full((target_h, target_w, 3), pad_color, dtype=np.uint8)
    dx = (target_w - nw) // 2
    dy = (target_h - nh) // 2
    canvas[dy:dy+nh, dx:dx+nw] = resized
    return canvas

tbs_letterbox = letterbox_resize(roi_straight, target_size=(224, 224))
print(f'Dimensi Citra Akhir Letterboxing: {tbs_letterbox.shape}')

# Visualisasi Tahapan Manipulasi Citra
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
# Konversi BGR ke RGB untuk visualisasi Matplotlib
axes[0].imshow(cv2.cvtColor(conveyor, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Citra Konveyor Asli & Bounding Box', fontsize=10, fontweight='bold')
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2)
axes[0].add_patch(rect)

axes[1].imshow(cv2.cvtColor(roi_straight, cv2.COLOR_BGR2RGB))
axes[1].set_title('2. ROI Buah Terotasi (-30 Deg)', fontsize=10, fontweight='bold')

axes[2].imshow(cv2.cvtColor(tbs_letterbox, cv2.COLOR_BGR2RGB))
axes[2].set_title('3. Penskalaan Letterbox (224x224)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('praktikum_manipulasi_citra_11_2.png', dpi=150)
plt.show()
